In [1]:
#import libraries
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

In [2]:
#define columns
column_names = ["age", "workclass", "fnlwgt", "education", "education_num", "marital_status", "occupation", 
               "relationship", "race", "sex", "capital_gain", "capital_loss", "hours_per_week", "native_country", "income"]

In [3]:
#load dataset
df_train = pd.read_csv('adult.data', names=column_names, skipinitialspace=True)
df_test = pd.read_csv('adult.test', names=column_names, skiprows=1, skipinitialspace=True)

In [4]:
#stadardize income column
df_train['income'] = df_train['income'].str.replace('.', '', regex=False)
df_test['income'] = df_test['income'].str.replace('.', '', regex=False)

In [5]:
print(df_train.dtypes)
print(df_test.dtypes)

age                int64
workclass         object
fnlwgt             int64
education         object
education_num      int64
marital_status    object
occupation        object
relationship      object
race              object
sex               object
capital_gain       int64
capital_loss       int64
hours_per_week     int64
native_country    object
income            object
dtype: object
age                int64
workclass         object
fnlwgt             int64
education         object
education_num      int64
marital_status    object
occupation        object
relationship      object
race              object
sex               object
capital_gain       int64
capital_loss       int64
hours_per_week     int64
native_country    object
income            object
dtype: object


In [6]:
#replace ? with Nan and drop missing values
for column in column_names:
    df_train[column].replace(' ?', pd.NA, inplace=True)
    df_test[column].replace(' ?', pd.NA, inplace=True)
    
df_train.dropna(inplace=True)
df_test.dropna(inplace=True)

In [7]:
#apply one-hot
categorical_cols = df_train.columns[df_train.dtypes == 'object']
df_train = pd.get_dummies(df_train, columns=categorical_cols, drop_first=True)
df_test = pd.get_dummies(df_test, columns=categorical_cols, drop_first=True)

In [8]:
#align test with train set
df_train, df_test = df_train.align(df_test, join='inner', axis=1)

In [9]:
#split data
X_train = df_train.drop('income_>50K', axis=1)
y_train = df_train['income_>50K']

X_test = df_test.drop('income_>50K', axis=1)
y_test = df_test['income_>50K']

In [10]:
#classifiers
classifiers = {
    'K-Nearest Neighbors': KNeighborsClassifier(),
    'Naive Bayes': GaussianNB(),
    'Support Vector Machine': SVC(),
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier(),
    'AdaBoost': AdaBoostClassifier(),
    'Gradient Boosting': GradientBoostingClassifier(),
    'Linear Discriminant Analysis': LinearDiscriminantAnalysis(),
    'Multi-layer Perceptron': MLPClassifier(max_iter=1000),
    'Logistic Regression': LogisticRegression(max_iter=1000)
}

In [11]:
#feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [12]:
#train and evaluate classifiers
results = {}
for name, clf in classifiers.items():
    if name in ['K-nearest Neighbors', 'Support Vector Machine', 'Multi-layer Perceptron']:
        clf.fit(X_train_scaled, y_train)
        predictions = clf.predict(X_test_scaled)
    else:
        clf.fit(X_train, y_train)
        predictions = clf.predict(X_test)
        
    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions, pos_label=True, zero_division=0)
    recall = recall_score(y_test, predictions, pos_label=True)
    f1 = f1_score(y_test, predictions, pos_label=True)
    auc = roc_auc_score(y_test, clf.predict_proba(X_test_scaled)[:, 1]) if hasattr(clf, 'predict_proba') else 'N/A'

    results[name] = {
        'Accuracy': round(accuracy, 2),
        'Precision': round(precision, 2),
        'Recall': round(recall, 2),
        'F1-score': round(f1, 2),
        'AUC': auc if auc != 'N/A' else 'N/A'
    }

#convert to df
results_df = pd.DataFrame(results).T

results_df

,Accuracy,Precision,Recall,F1-score,AUC
K-Nearest Neighbors,0.78,0.55,0.32,0.4,0.5
Naive Bayes,0.8,0.64,0.3,0.41,0.844635
Support Vector Machine,0.85,0.74,0.57,0.64,N/A
Decision Tree,0.81,0.6,0.62,0.61,0.499323
Random Forest,0.85,0.72,0.61,0.66,0.817982
AdaBoost,0.86,0.75,0.61,0.67,0.827203
Gradient Boosting,0.87,0.79,0.61,0.69,0.835488
Linear Discriminant Analysis,0.84,0.72,0.56,0.63,0.834327
Multi-layer Perceptron,0.83,0.64,0.61,0.62,0.869996
Logistic Regression,0.85,0.72,0.56,0.63,0.861252
